# QC A3 Parity Pilot

This notebook validates the Investintell A3 path inside QuantConnect Research without using QC FRED or returns as an A3 objective. It consumes immutable Investintell L2/revision-uncertainty/config artifacts and compares normalized outputs against the local worker bundle.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import platform
import sys
import time
import tracemalloc

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qc_a3_core import A3ParityConfig, materialize_harness_source_from_manifest, read_json, run_parity, write_json


## Object Store contract

Upload every file listed in `object_store_manifest.json` to its matching Object Store key. The core A3 run must read only these Investintell artifacts. QC History/FRED is intentionally absent from this notebook.

In [ ]:
OBJECT_STORE_MANIFEST_KEY = "investintell/a3/qc-a3-parity/25375bb/10198d7603036c3327ac9e67/object_store_manifest.json"

def maybe_quantbook_object_store():
    try:
        from QuantConnect.Research import QuantBook
    except Exception:
        return None
    qb = QuantBook()
    return getattr(qb, "object_store", getattr(qb, "ObjectStore", None))

object_store = maybe_quantbook_object_store()

def object_store_file_path(key: str) -> Path | None:
    if object_store is None:
        return None
    if hasattr(object_store, "get_file_path"):
        return Path(object_store.get_file_path(key))
    if hasattr(object_store, "GetFilePath"):
        return Path(object_store.GetFilePath(key))
    raise RuntimeError("QuantConnect Object Store does not expose get_file_path/GetFilePath")

manifest_path = Path("object_store_manifest.json")
if not manifest_path.exists():
    manifest_path = object_store_file_path(OBJECT_STORE_MANIFEST_KEY) or Path("_tmp_qc_a3_parity/object_store_manifest.json")

manifest = read_json(manifest_path)
manifest["_local_bundle_dir"] = str(manifest_path.parent)
assert manifest["bridge_scope"] == "qc_research_parity_only"
assert manifest["runtime_activation"] is False
assert manifest["a3_status"] == "open_macro_v03"
assert manifest["upload_policy"]["parquet_upload_allowed"] is False
materialize_harness_source_from_manifest(manifest, object_store_file_path, project_root=PROJECT_ROOT)
manifest["selected"]


In [ ]:
tracemalloc.start()
total_started = time.perf_counter()
load_started = time.perf_counter()

bundle_dir = manifest_path.parent

def artifact_path(name: str) -> Path:
    item = manifest["object_files"][name]
    path = object_store_file_path(item["object_store_key"])
    if path is not None:
        return path
    return bundle_dir / item["relative_path"]

feature_manifest_path = artifact_path("feature_manifest")
revision_uncertainty_manifest_path = artifact_path("revision_uncertainty_manifest")
config_catalog_path = artifact_path("config_catalog_normalized")
macro_l2_npz_path = artifact_path("macro_l2_union_numeric")
revision_uncertainty_npz_path = artifact_path("revision_uncertainty_numeric")
object_store_load_seconds = time.perf_counter() - load_started

results_dir = Path("results")
config = A3ParityConfig(
    feature_manifest=feature_manifest_path,
    revision_uncertainty_manifest=revision_uncertainty_manifest_path,
    config_catalog=config_catalog_path,
    a32_grid_dir=feature_manifest_path.parent,
    output_dir=results_dir,
    macro_l2_npz=macro_l2_npz_path,
    revision_uncertainty_npz=revision_uncertainty_npz_path,
    a31_name=manifest["selected"]["a31_config_name"],
    a32_name=manifest["selected"]["a32_config_name"],
    worker_commit=manifest["worker_commit"],
)
scoring_started = time.perf_counter()
report = run_parity(config)
scoring_seconds = time.perf_counter() - scoring_started
current_bytes, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
total_seconds = time.perf_counter() - total_started
report["external_macro_access"] = False
report["object_store_prefix_immutable"] = manifest["object_store_prefix_immutable"]
report["timings"] = {
    "object_store_load_seconds": object_store_load_seconds,
    "scoring_time_seconds": scoring_seconds,
    "metrics_time_seconds": None,
    "total_wall_clock_seconds": total_seconds,
}
report["memory"] = {"peak_tracemalloc_bytes": peak_bytes, "current_tracemalloc_bytes": current_bytes}
write_json(results_dir / "qc_cloud_parity_report.json", report)
report["comparison"]


In [ ]:
assert report["runtime_activation"] is False
assert report["a4_status"] == "harness_ready_provisional_A3"
assert report["a5_status"] == "blocked"
assert report["external_macro_access"] is False
assert report["runtime_row_count"] == manifest["expected"]["runtime_row_count"]
assert report["counterfactual_row_count"] == manifest["expected"]["counterfactual_row_count"]
assert report["metric_row_count"] == manifest["expected"]["metric_row_count"]
assert report["runtime_replay_logical_hash"] == manifest["expected"]["macro_runtime_replay_logical_hash"]
assert report["counterfactual_replay_logical_hash"] == manifest["expected"]["macro_counterfactual_replay_logical_hash"]
assert report["metric_rows_logical_hash"] == manifest["expected"]["macro_metric_rows_logical_hash"]
assert report["comparison"].get("mismatch_count", 0) == 0

def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

environment = {
    "research_node": os.environ.get("QC_NODE_TYPE") or os.environ.get("QC_NODE_NAME"),
    "lean_version": os.environ.get("LEAN_VERSION"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "numpy_version": package_version("numpy"),
    "pandas_version": package_version("pandas"),
    "object_store_prefix_immutable": manifest["object_store_prefix_immutable"],
    "timings": report["timings"],
    "memory": report["memory"],
}
write_json(results_dir / "qc_cloud_environment.json", environment)
print(report["runtime_replay_logical_hash"])
print(report["metric_rows_logical_hash"])
